**Prova Python:**

Para los que van a la prueba 2, puntos que pueden repasar:
- se entregara la bbdd para cargar en mysql tal cual la prueba 1
- conexión a mysql: cualquiera, lo único que cambiará será el nombre de la nueva bbdd
- guardar tablas en df
- mostrar información de un df: primeras filas, últimas filas, tipos de datos, cantidad de filas, cantidad de columnas, etc
- crear una nueva columna partiendo de alguna/s ya conocidas, tipo: monto total desde cantidad y precio, 1 y 0 sin cumple alguna condición; bajo, medio o alto si cumple alguna condición, etc
- gráfica de una variable categórica
- gráfica de una variable numérica

Lo que sigue pueden probar tomando las variables desde la misma tabla y desde tablas diferentes:
- gráfica que relacione 2 variables numéricas
- gráfica de 1 variable numérica y 1 categórica
- gráfica de 2 variables categóricas y 1 numérica
- 1  gráfica que quieran en power bi
- eliminar la base de datos y cerrar conexiones

# Llibreries

In [34]:
# Connexió Python - MySQL
import mysql.connector
from mysql.connector import Error

from dotenv import load_dotenv  
import os

load_dotenv()

import pandas as pd

# Visualitzacions i datetime
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import matplotlib.ticker as ticker
import datetime as dt
from datetime import datetime, date

# MySQL Python

 ## Connectar amb MySQL

In [2]:
# Document amb environment (.env) on hi ha les dades del host, user i password
try:
    connection = mysql.connector.connect(host=os.getenv("DB_HOST"),
                                         user=os.getenv('DB_USER'),
                                         password=os.getenv('DB_PASSWORD'))
    if connection.is_connected():
        cursor = connection.cursor(buffered=True)
        cursor.execute("show databases;")
        record = cursor.fetchall()
        print("Available databases: ", record)

except Error as e:
    print("Error while connecting to MySQL", e)

Available databases:  [('bootcamp_exam',), ('information_schema',), ('mysql',), ('performance_schema',), ('s4_transactions',), ('sys',), ('transactions',)]


## Crear bbdd per posar les dades

create_database = """
CREATE DATABASE IF NOT EXISTS bootcamp_exam;
"""

cursor.execute(create_database)
cursor.execute("show databases;")
record = cursor.fetchall()
print("Available databases: ", record)


Connectar a la bbdd creada / es pot fer amb USE i així no reconnectar (2 maneres, una tornes a connectar i amb l'altra només actives la bbdd)

cursor.execute("USE bootcamp_exam")

try:
    connection = mysql.connector.connect(host=os.getenv("DB_HOST"),
                                         database='bootcamp_exam',
                                         user=os.getenv('DB_USER'),
                                         password=os.getenv('DB_PASSWORD'))
    if connection.is_connected():
        cursor = connection.cursor(buffered=True)
        cursor.execute("show tables;")
        record = cursor.fetchall()
        print("Available databases: ", record)

except Error as e:
    print("Error while connecting to MySQL", e)

Create table and insert values (be aware of commit!) and show table, with error managing

Més d'una query dintre del mateix execute

## Obrir un document i corre totes les queries

In [22]:
with open("script.txt", "r", encoding="utf-8") as f:
    sql_script = f.read()

try:
    for statement in sql_script.split(";"): # Divideix tot l'script en les diverses queries
        statement = statement.strip() # Neteja els possibles '"' que queden a l'statement
        if statement:
            try: 
                cursor.execute(statement)
                print("Number of rows affected", cursor.rowcount)
            except mysql.connector.Error as error: # Ens diu quina query té un error
                print("❌ ERROR EN AQUESTA SENTÈNCIA:")
                print(statement)
                print("Error:", error)
                break   

    connection.commit() # guarda les dades a la bbdd, si no estigués no es guarden i sembla que no les hagis pujat

    cursor.execute("SELECT DATABASE()")
    database = cursor.fetchall()

    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()

    print(f'Current tables in {database}": {tables}')
except mysql.connector.Error as error:
    print(f'Error: {error}')

Number of rows affected 1
Number of rows affected 0
Number of rows affected 0
Number of rows affected 8
Number of rows affected 0
Number of rows affected 6
Number of rows affected 0
Number of rows affected 10
Number of rows affected 0
Current tables in [('bootcamp_exam',)]": [('orders',), ('products',), ('users',)]


## Carregar les queries de forma manual

sql_operation = """
CREATE DATABASE bootcamp_exam;
USE bootcamp_exam;

CREATE TABLE users (
id INT PRIMARY KEY,
name VARCHAR(50),
age INT,
country VARCHAR(50)
);

INSERT INTO users VALUES
(1, 'Anna', 25, 'Spain'),
(2, 'Marc', 32, 'France'),
(3, 'Laura', 45, 'Spain'),
(4, 'John', 28, 'USA'),
(5, 'Sofia', 35, 'Italy'),
(6, 'Carlos', 22, 'Spain'),
(7, 'Emma', 41, 'Germany'),
(8, 'Lucas', 30, 'France');

CREATE TABLE products (
id INT PRIMARY KEY,
product_name VARCHAR(50),
price FLOAT,
category VARCHAR(50)
);

INSERT INTO products VALUES
(1, 'Laptop', 900, 'Electronics'),
(2, 'Phone', 600, 'Electronics'),
(3, 'Tablet', 300, 'Electronics'),
(4, 'Chair', 120, 'Furniture'),
(5, 'Desk', 250, 'Furniture'),
(6, 'Headphones', 80, 'Electronics');

CREATE TABLE orders (
id INT PRIMARY KEY,
user_id INT,
product_id INT,
quantity INT,
order_date DATE,
FOREIGN KEY (user_id) REFERENCES users(id),
FOREIGN KEY (product_id) REFERENCES products(id)
);

INSERT INTO orders VALUES
(1, 1, 1, 1, '2024-01-10'),
(2, 2, 2, 2, '2024-01-12'),
(3, 3, 3, 1, '2024-01-15'),
(4, 4, 4, 4, '2024-01-18'),
(5, 5, 5, 1, '2024-01-20'),
(6, 6, 6, 3, '2024-01-22'),
(7, 1, 2, 1, '2024-01-25'),
(8, 2, 3, 2, '2024-01-28'),
(9, 3, 1, 1, '2024-02-01'),
(10, 4, 5, 1, '2024-02-05');
"""

try:
    for statement in sql_operation.split(";"):
        statement = statement.strip()
        if statement:
            cursor.execute(statement)

    connection.commit()
    
    cursor.execute("SHOW TABLES")
    print(cursor.fetchall())
except mysql.connector.Error as error:
    print(f'Error: {error}')


Dividir manualment les queries. Sobretot utilitzar el commit de connection per a carregar bé les dades (INSERT)

query = """
CREATE TABLE users (
id INT PRIMARY KEY,
name VARCHAR(50),
age INT,
country VARCHAR(50)
);

INSERT INTO users VALUES
(1, 'Anna', 25, 'Spain'),
(2, 'Marc', 32, 'France'),
(3, 'Laura', 45, 'Spain'),
(4, 'John', 28, 'USA'),
(5, 'Sofia', 35, 'Italy'),
(6, 'Carlos', 22, 'Spain'),
(7, 'Emma', 41, 'Germany'),
(8, 'Lucas', 30, 'France');
"""

cursor.execute(query)

connection.commit()

## DROP Table

query = """
DROP TABLE users;
"""

cursor.execute(query)

## Pandas

### Transformar taules a diccionari de df

Veure les taules que hi ha a la bbdd

In [24]:
cursor.execute("Show tables;")

tables = cursor.fetchall()

tables

[('orders',), ('products',), ('users',)]

Transformació

In [25]:
def select_table(table_name: str):
    query = f"SELECT * FROM {table_name}"
    cursor.execute(query)
    
    data = cursor.fetchall()
    columns = [col[0] for col in cursor.description]

    df = pd.DataFrame(data, columns=columns)
    return df

In [29]:
tables_df = {}

for x in tables:
    table_name = x[0]
    print(f'Taula transformada: {table_name}')
    tables_df[table_name] = select_table(table_name)

Taula transformada: orders
Taula transformada: products
Taula transformada: users


In [30]:
df_orders = tables_df['orders'].copy()
df_products = tables_df['products'].copy()
df_users = tables_df['users'].copy()

Query model per utilitzar.

query = """
SELECT u.country, o.amount
FROM orders o
JOIN users u ON o.user_id = u.id
"""

cursor.execute(query)

data = cursor.fetchall()  
columns = [col[0] for col in cursor.description]

df = pd.DataFrame(data, columns=columns)

## DROP Database

In [ ]:
query = """
DROP DATABASE bootcamp_exam;
"""

cursor.execute(query)

## Tancar connexió

In [31]:
cursor.close()
connection.close()

if not connection.is_connected():
    print("Connexió tancada correctament ✅")

Connexió tancada correctament ✅


In [32]:
try:
    cursor.execute("SELECT 1")
except:
    print("Cursor tancat o connexió no disponible ✅")

Cursor tancat o connexió no disponible ✅


# Visualitzacions

## Format

In [35]:
sns.set_style("dark")
sns.set_palette("pastel")

# Pàgines web

INSERT INTO
- [Python MySQL Insert Into Table](https://www.w3schools.com/python/python_mysql_insert.asp)
- https://www.youtube.com/watch?v=7Pa3sWBzYkA

# Extres

cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
    id INT PRIMARY KEY,
    name VARCHAR(50),
    age INT,
    country VARCHAR(50)
    );
    """)

## Execute many

In [ ]:
try:
    create_table = """
    CREATE TABLE IF NOT EXISTS users (
    id INT PRIMARY KEY,
    name VARCHAR(50),
    age INT,
    country VARCHAR(50)
    );
    """

    sql = """INSERT INTO users (id, name, age, country)
            VALUES (%s, %s, %s, %s)
            """
    val = [
    (1, 'Anna', 25, 'Spain'),
    (2, 'Marc', 32, 'France'),
    (3, 'Laura', 45, 'Spain'),
    (4, 'John', 28, 'USA'),
    (5, 'Sofia', 35, 'Italy'),
    (6, 'Carlos', 22, 'Spain'),
    (7, 'Emma', 41, 'Germany'),
    (8, 'Lucas', 30, 'France')
    ]

    cursor.execute(create_table)
    cursor.executemany(sql, val)

    connection.commit()

    cursor.execute("SHOW TABLES")
    print(cursor.fetchall())
except mysql.connector.Error as error:
    print(f'Error: {error}')

## Pandasql

Pandasql is not supported by the MySQL connector.

In [ ]:
df_orders = pd.read_sql('SELECT * FROM orders', con=connection)
df_orders.head(2)

C:\Users\sabin\AppData\Local\Temp\ipykernel_5612\3631347618.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_orders = pd.read_sql('SELECT * FROM orders', con=connection)


,id,user_id,product_id,quantity,order_date
0,1,1,1,1,2024-01-10
1,2,2,2,2,2024-01-12


In [ ]:
cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

tables_df = {}

for t in tables:
    table_name = t[0]
    tables_df[table_name] = pd.read_sql(f"SELECT * FROM {table_name}", conn)